<h1 style="color:#1f77b4;">SARIMAX BANXICO API - EXAM</h1>

 `Autoras:` 
 - Sarah Lucia Beltrán Gutiérrez
 - Aissa Berenice
 - Isabel Valladolid

 `Fecha:` 04/03/2026

 `Materia:` Non Linear Models

 `Profesor:` Pedro Martinez

---

<h2 style="color:#1f77b4;">Introduccion</h2>

En este proyecto se desarrolla un modelo `SARIMAX` para analizar y predecir el comportamiento del tipo de cambio **peso mexicano–dólar estadounidense** (FIX), utilizando datos oficiales obtenidos mediante la API del Banco de México. A partir de una serie temporal diaria, se busca evaluar la capacidad predictiva del modelo en el corto plazo, específicamente para el periodo del **5 al 13 de marzo de 2026**. El trabajo integra la descarga y procesamiento de datos, la justificación del modelo elegido y la evaluación de su desempeño mediante el error `MAPE`, con el fin de medir qué tan eficaz resulta `SARIMA` para anticipar movimientos del

---

In [23]:
# librerias
from dotenv import load_dotenv
import os
import requests
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import grangercausalitytests
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_ccf
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
# from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
# from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.stattools import adfuller, acf, pacf

In [2]:
# token desde .env
load_dotenv()
TOKEN = os.getenv("BANXICO_TOKEN")

In [3]:
# llamamos api :p
url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF43718/datos"
headers = {"Bmx-Token": TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()
data = response.json()

In [4]:
# df
datos = data["bmx"]["series"][0]["datos"]
df = pd.DataFrame(datos)   
df.head()

,fecha,dato
0,12/11/1991,3.0735
1,13/11/1991,3.0712
2,14/11/1991,3.0718
3,15/11/1991,3.0684
4,18/11/1991,3.0673


In [5]:
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
df["dato"] = pd.to_numeric(df["dato"], errors="coerce")

# fecha como indice
df.set_index("fecha", inplace=True)
df.sort_index(inplace=True)

<h2 style="color:#1f77b4;">Visualizacion</h2>

In [6]:
fig = px.line(
    df,
    x=df.index,
    y="dato",
    title="Serie de Tiempo: Tipo de Cambio FIX (MXN/USD)"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Tipo de cambio FIX",
)

fig.show()

En la gráfica anterior se muestra la serie de tiempo de la base de datos de Banxico desde 1995 hasta 2026, a continuación se realizara el "corte" para que la prediccion no se ensucie con datos super viejos. Se tomara desde enero 2016.

In [36]:
import holidays

In [37]:
df = df.loc['2015-01-01':]

In [41]:
# Creamos la columna llena de ceros (variable dummy)
df['Holiday_Dummy'] = 0
# Descargamos los holidays de USA de los años que hay en TSA
years = df.index.year.unique()
mx_holidays = holidays.MX(years=years)
us_holidays = holidays.US(years=years)


# Lógica: Marcamos con 1 si la fecha cae en una ventana ALGUNOS días
for date in df.index:
    # Ventana de días
    window = pd.date_range(start=date - pd.Timedelta(days=1),
                           end=date + pd.Timedelta(days=1))

    # Si algún día de la ventana es festivo, marcamos el día actual como 1
    if any(d in mx_holidays for d in window):
        df.loc[date, 'Holiday_Dummy'] = 1
    elif any(d in us_holidays for d in window):
        df.loc[date, 'Holiday_Dummy'] = 1  

In [42]:
df['Holiday_Dummy'].value_counts()


Holiday_Dummy
0    2267
1     289
Name: count, dtype: int64

In [43]:
fig = px.line(
    df,
    x=df.index,
    y="dato",
    title="Serie de Tiempo: Tipo de Cambio FIX (MXN/USD) – Datos Recientes"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Tipo de cambio FIX",
)

fig.show()

<h2 style="color:#1f77b4;">Análisis para el (p,d,q) y (P,D,Q,s)</h2> 

In [44]:

def check_stationarity(series, title="Serie Original"):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'Estadístico ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"¿Es estacionaria? {'SÍ' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. Revisamos la serie original
check_stationarity(df['dato'], "Nivel Original")

# 2. Aplicamos Primera Diferencia (d=1)
df_diff = df['dato'].diff()

# 3. Revisamos la serie diferenciada
check_stationarity(df_diff, "Primera Diferencia (d=1)")

# Creamos una figura con 2 columnas (Subplots)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Serie Original (No Estacionaria)", "Serie Diferenciada (Estacionaria d=1)")
)

# Gráfico 1: Serie Original
fig.add_trace(
    go.Scatter(x=df.index, y=df['dato'], name='Original'),
    row=1, col=1
)

# Gráfico 2: Serie Diferenciada
fig.add_trace(
    go.Scatter(x=df_diff.index, y=df_diff, name='Diferenciada'),
    row=1, col=2
)

# Ajustes de diseño
fig.update_layout(
    title_text="Comparativa: Efecto de la Diferenciación",
    showlegend=False, # Ocultamos leyenda
    height=500
)

fig.show()

ADF Test: Nivel Original
Estadístico ADF: -2.9603
p-value: 0.0388
¿Es estacionaria? SÍ

ADF Test: Primera Diferencia (d=1)
Estadístico ADF: -47.6731
p-value: 0.0000
¿Es estacionaria? SÍ



In [47]:

# @title Graficamos ACF y PACF

ts_analysis = df_diff.dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

# Resaltar lags estacionales (7, 14, 21, 28) con líneas verticales rojas tenues
for i in [7, 14, 21, 28]:
    fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()

In [ ]:
# Tranformamos pasajeros a log
df['Log_Datos'] = np.log(df['dato'])

# Diferenciación Estacional (Lag 7 para datos diarios)
# Restamos a hoy el valor de hace una semana
df['Diff_7'] = df['Log_Passengers'].diff(7).dropna() # calcular la diferencia del valor 7 (hoy menos lo que pasará en 7 días)

print("Resultados de Estacionariedad:")
run_adf(df['Diff_7'], "Log Pasajeros (Diferencia Semanal)")

# Gráficas ACF y PACF
# Esto nos ayuda a elegir p y q del modelo
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
# Usamos la serie diferenciada porque es la que entra al modelo
plot_acf(df['Diff_7'].dropna(), lags=40, ax=ax1, title="Autocorrelación (ACF) - Patron MA(q)")
plot_pacf(df['Diff_7'].dropna(), lags=40, ax=ax2, title="Autocorrelación Parcial (PACF) - Patron AR(p)")
plt.tight_layout()
plt.show()

# para la D hacer la prueba de digui fuller con el lag de 7

# no siempre el que nuestra seri sea estacionaria significa que será 0


KeyError: 'Log_Passengers'

Cuando la serie sale asi es porque el tipo de cambio esta super parecido, como rl tipo de cambio no cambia mucho entonces sale asi porque cambia decimales :P

In [31]:
# como sabe si nuestro coeficiente será SARIMAX

print("\n" + "="*50)
print("PRUEBA DE CAUSALIDAD DE GRANGER")
print("Hipótesis Nula (H0): Los días festivos NO causan cambios en el tráfico.")
print("Serie analizada: Log_Passengers (Sin Diff Estacional)")
print("="*50)

# Preparamos el DataFrame para la prueba
# La primera columna debe ser la serie TARGET
# La segunda columna debe ser la serie AYUDA
granger_data = df[['Log_Passengers', 'Holiday_Dummy']].dropna()

# Ejecutamos la prueba para los primeros 7 lags (días)
# verbose=True imprime los resultados detallados

granger_results = grangercausalitytests(granger_data, maxlag=7, verbose=True)


print("\n" + "="*50)
print("GUÍA DE INTERPRETACIÓN:")
print("Busca la fila 'ssr_ftest' y su columna 'p-value'.")
print("Si p < 0.05, estadísticamente la variable 'Holiday_Dummy' ayuda a predecir 'Log_Passengers'.")
print("="*50)


PRUEBA DE CAUSALIDAD DE GRANGER
Hipótesis Nula (H0): Los días festivos NO causan cambios en el tráfico.
Serie analizada: Log_Passengers (Sin Diff Estacional)


KeyError: "None of [Index(['Log_Passengers', 'Holiday_Dummy'], dtype='str')] are in the [columns]"

# EXPERIMENTOS PARA EL MODELADO